**Contexte**


Une entreprise souhaite développer un système d’intelligence artificielle capable de reconnaître
automatiquement le type de déchet présent sur une photographie afin d'améliorer le tri des
déchets.
Le modèle devra classer chaque image dans l'une des catégories suivantes :

``cardboard`` : cartons ondulés, cartons plats, …

``plastic`` : bouteilles, emballages plastiques...

``paper`` : feuilles, journaux...

``glass`` : bouteilles et objets en verre...

``metal`` : canettes, boîtes métalliques...

``trash`` : emballages bonbons, tasses jetables, ...

Le problème est que les images collectées proviennent de plusieurs sources. Elles ne sont donc pas
homogènes : dimensions différentes ; formats différents ; images RGB et grayscale ; certaines images
sont trop petites ; certaines images sont corrompues ; quelques images sont vides ; images
dupliquées ; quelques images placées dans le mauvais dossier ; classes déséquilibrées.
L'objectif de l'atelier est donc de construire un jeu de données images propre et homogène, prêt à
être utilisé pour entraîner un modèle de Machine Learning ou de Deep Learning.

**Objectifs pédagogiques**

À la fin de l'atelier, l'apprenant devra être capable de :
1) explorer un dataset d'images ;
2) détecter les images problématiques ;
3) détecter les différences de résolution ;
4) détecter les différences de nombre de canaux ;
5) identifier les images trop petites ;
6) détecter les doublons ;
7) identifier les classes déséquilibrées ;
8) redimensionner les images ;
9) normaliser les valeurs des pixels ;
10) uniformiser les canaux ;
11) appliquer de la data augmentation

In [5]:
import pandas as pd
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
import numpy as np

# alors pour cette atelier je vais utiliser les variables os path pour mieux gérer les chemins de fichiers et dossiers

import os
from pathlib import Path

# Chemins du projet : on utilise Path (plus lisible/robuste que des chaînes de caractères concaténées)
RAW_DIR = Path("../data/raw")
CLEANED_DIR = Path("../data/cleaned")
REPORTS_DIR = Path("../reports")
CLASSES = sorted([d.name for d in RAW_DIR.iterdir() if d.is_dir()])

print("Les classes détectées :", CLASSES)

Les classes détectées : ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']


**Partie 1 – Exploration du dataset**

Développer un programme Python capable de récupérer, pour chaque image, son nom, sa classe,
son format, son mode, sa largeur, sa hauteur, l’écart-type de ses pixels, son nombre de canaux et sa
taille.

``NB :`` prendre en charge aussi les fichiers corrompus

In [ ]:
def explorer_image(chemin, classe):
    """
    Ouvre une image et retourne un dictionnaire avec toutes ses caractéristiques.
    Si l'image est corrompue (illisible), retourne un dictionnaire avec des valeurs manquantes
    et corrompue=True, SANS lever d'exception (grâce au try/except).
    """
    fiche = {
        "nom": chemin.name,
        "classe": classe,
        "chemin": str(chemin),
    }


    try:
        fiche["taille_octets"] = chemin.stat().st_size
    except OSError:
        fiche["taille_octets"] = None

    try:
        with Image.open(chemin) as img:
            img.load()

            fiche["corrompue"] = False
            fiche["format"] = img.format               
            fiche["mode"] = img.mode                    
            fiche["largeur"], fiche["hauteur"] = img.size
            fiche["nb_canaux"] = len(img.getbands())    

            arr = np.array(img.convert("L"))
            fiche["std_pixels"] = float(arr.std())

    except Exception as e:
        fiche["corrompue"] = True
        fiche["format"] = None
        fiche["mode"] = None
        fiche["largeur"] = None
        fiche["hauteur"] = None
        fiche["nb_canaux"] = None
        fiche["std_pixels"] = None
        fiche["erreur"] = str(e)

    return fiche


fiches = []
for classe in CLASSES:
    dossier_classe = RAW_DIR / classe
    for chemin_image in sorted(dossier_classe.iterdir()):
        fiches.append(explorer_image(chemin_image, classe))

audit = pd.DataFrame(fiches)
print(f"{len(audit)} images explorées au total.")
audit.head(10)


1032 images explorées au total.


,nom,classe,chemin,taille_octets,corrompue,format,mode,largeur,hauteur,nb_canaux,std_pixels,erreur
0,cardboard1.jpg,cardboard,..\data\raw\cardboard\cardboard1.jpg,17333,False,JPEG,RGB,512.0,384.0,3.0,31.875892,NaN
1,cardboard10.jpg,cardboard,..\data\raw\cardboard\cardboard10.jpg,21683,False,JPEG,RGB,512.0,384.0,3.0,38.799907,NaN
2,cardboard100.jpg,cardboard,..\data\raw\cardboard\cardboard100.jpg,14884,False,JPEG,RGB,512.0,384.0,3.0,44.498372,NaN
3,cardboard101.jpg,cardboard,..\data\raw\cardboard\cardboard101.jpg,14289,False,JPEG,RGB,512.0,384.0,3.0,68.561937,NaN
4,cardboard102.jpg,cardboard,..\data\raw\cardboard\cardboard102.jpg,18015,False,JPEG,RGB,512.0,384.0,3.0,45.320633,NaN
5,cardboard103.jpg,cardboard,..\data\raw\cardboard\cardboard103.jpg,21104,False,JPEG,RGB,512.0,384.0,3.0,35.713359,NaN
6,cardboard104.jpg,cardboard,..\data\raw\cardboard\cardboard104.jpg,17225,False,JPEG,RGB,512.0,384.0,3.0,31.949912,NaN
7,cardboard105.jpg,cardboard,..\data\raw\cardboard\cardboard105.jpg,24417,False,JPEG,RGB,512.0,384.0,3.0,42.665779,NaN
8,cardboard106.jpg,cardboard,..\data\raw\cardboard\cardboard106.jpg,26388,False,JPEG,RGB,512.0,384.0,3.0,52.755686,NaN
9,cardboard107.jpg,cardboard,..\data\raw\cardboard\cardboard107.jpg,25368,False,JPEG,RGB,512.0,384.0,3.0,34.045471,NaN


j'ai maintenant un tableau `audit` avec **une ligne par image** et toutes les informations
demandées. C'est ce tableau qui va me servir de base à absolument toutes les parties suivantes
(Parties 2 à 8) : plutôt que de reparcourir le dossier d'images à chaque question, on filtre
simplement ce DataFrame.

**Partie 2 – Détecter les images corrompues**

**Définition d'image corrompu**

C'est un fichier qui *prétend* être une image (souvent via son extension `.jpg`, `.png`...) mais
dont les données sont endommagées : en-tête invalide, fichier tronqué (coupé avant la fin), ou
contenu binaire aléatoire. Pillow lève alors une exception quand on essaie de le lire.

In [11]:
def est_corrompue(chemin):
    """
    Retourne True si l'image ne peut pas être ouverte ET intégralement décodée par Pillow.

    On fait DEUX passes volontairement :
      1) img.verify() : une vérification rapide de l'intégrité du fichier (structure interne),
         sans décoder complètement les pixels.
      2) Image.open(...).load() : un rechargement complet avec décodage de tous les pixels.

    Pourquoi deux passes et pas une seule ?
    Parce que .verify() a un effet de bord : une fois appelée, l'objet Image ne peut plus être
    réutilisé pour d'autres opérations. Et surtout, certains fichiers légèrement endommagés
    passent .verify() (la structure globale semble correcte) mais échouent au décodage complet
    des pixels avec .load(). Faire les deux donne une détection plus fiable qu'une seule des deux.
    """
    try:
        with Image.open(chemin) as img:
            img.verify()
        with Image.open(chemin) as img:
            img.load()
        return False
    except Exception:
        return True


audit["corrompue_v2"] = audit["chemin"].apply(lambda p: est_corrompue(Path(p)))
assert (audit["corrompue"] == audit["corrompue_v2"]).all(), "Incohérence entre les deux détections !"
audit = audit.drop(columns=["corrompue_v2"])

images_corrompues = audit[audit["corrompue"]]
print(f"Nombre d'images corrompues détectées : {len(images_corrompues)}")
images_corrompues[["classe", "nom", "erreur"]]

Nombre d'images corrompues détectées : 6


,classe,nom,erreur
147,cardboard,cardboard83.jpg,cannot identify image file '..\\data\\raw\\car...
326,glass,glass74.jpg,Truncated File Read
446,metal,metal48.jpg,cannot identify image file '..\\data\\raw\\met...
633,paper,paper213.jpg,cannot identify image file '..\\data\\raw\\pap...
791,plastic,plastic13.jpg,cannot identify image file '..\\data\\raw\\pla...
1004,trash,trash3.jpg,cannot identify image file '..\\data\\raw\\tra...
